In [1]:
!pip -q install --upgrade pip
!pip -q install unsloth transformers datasets trl accelerate peft bitsandbytes kagglehub pandas

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.8/1.8 MB 86.4 MB/s eta 0:00:00


In [2]:
MODEL_NAME = "Qwen/Qwen2-7B-Instruct"
MAX_SEQ_LENGTH = 2048
LOAD_IN_4BIT = False # Change this to False for LoRA, True for QLoRA
OUTPUT_DIR = "sft-lora"
NUM_TRAIN_EPOCHS = 5

SYSTEM_PROMPT = "You are a weather forecasting assistant. You should only output in JSON format"
TRAIN_SPLIT = 0.8
FEATURE_COLUMNS = ['date', 'temp_max', 'temp_min', 'precipitation', 'wind', 'weather']
WEATHER_LABELS = ["drizzle", "rain", "sun", "snow", "fog"]
EXTERNAL_DATASET = "seattle-weather.csv"
WINDOW_SIZES = [7, 14, 21, 28]


In [3]:
from pathlib import Path
import json
import random
from copy import deepcopy
import pandas as pd
import kagglehub

DATA_DIR = Path("/content/weather_data")
DATA_DIR.mkdir(parents=True, exist_ok=True)
JSONL_OUT_PATH = DATA_DIR / "seattle_weather_chat.jsonl"
TRAIN_OUT_PATH = DATA_DIR / "seattle_weather_chat_train.jsonl"
EVAL_OUT_PATH = DATA_DIR / "seattle_weather_chat_eval.jsonl"
PREF_TRAIN_OUT_PATH = DATA_DIR / "seattle_weather_pref_train.jsonl"
PREF_EVAL_OUT_PATH = DATA_DIR / "seattle_weather_pref_eval.jsonl"

def format_day(row, day_offset):
    return (
        f"Day {day_offset} ({row['date'].date()}): "
        f"temp_max={row['temp_max']}, temp_min={row['temp_min']}, "
        f"precipitation={row['precipitation']}, wind={row['wind']}, "
        f"weather={row['weather']}"
    )

def format_target(row):
    target = {
        "temp_max": float(row['temp_max']),
        "temp_min": float(row['temp_min']),
        "precipitation": float(row['precipitation']),
        "wind": float(row['wind']),
        "weather": row['weather'],
    }
    return json.dumps(target, separators=(",", ":"))

def pull_data(dataset_name, output_directory):
    output_directory.mkdir(parents=True, exist_ok=True)
    return kagglehub.dataset_download(
        dataset_name,
        output_dir=str(output_directory),
        force_download=True,
    )

def format_data(path_to_dataset):
    df = pd.read_csv(path_to_dataset)
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values('date').drop_duplicates(subset=['date']).reset_index(drop=True)
    df = df.dropna().reset_index(drop=True)
    return df

def build_window_samples(data_frame, window_size):
    samples = []
    for t in range(window_size, len(data_frame)):
        window = data_frame.iloc[t - window_size:t]
        target_row = data_frame.iloc[t]

        lines = [format_day(window.iloc[i], i - window_size) for i in range(window_size)]
        user = (
            f"Given the last {window_size} days of Seattle weather, predict tomorrow.\n"
            + "\n".join(lines)
            + "\nReturn JSON with keys: temp_max, temp_min, precipitation, wind, weather."
        )

        samples.append({
            "messages": [
                {"role": "system", "content": SYSTEM_PROMPT},
                {"role": "user", "content": user},
                {"role": "assistant", "content": format_target(target_row)},
            ]
        })
    return samples

def split_samples(samples):
    if len(samples) < 2:
        return samples, []
    split_index = int(len(samples) * TRAIN_SPLIT)
    split_index = min(max(split_index, 1), len(samples) - 1)
    return samples[:split_index], samples[split_index:]

def write_jsonl(path, samples):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open("w", encoding="utf-8") as file:
        for sample in samples:
            file.write(json.dumps(sample) + "\n")

def write_samples(data_frame):
    all_samples, train_samples, eval_samples = [], [], []
    for window_size in WINDOW_SIZES:
        window_samples = build_window_samples(data_frame, window_size)
        split_train, split_eval = split_samples(window_samples)
        all_samples.extend(window_samples)
        train_samples.extend(split_train)
        eval_samples.extend(split_eval)

    write_jsonl(JSONL_OUT_PATH, all_samples)
    write_jsonl(TRAIN_OUT_PATH, train_samples)
    write_jsonl(EVAL_OUT_PATH, eval_samples)

    print(f"Total samples: {len(all_samples)}")
    print(f"Train samples: {len(train_samples)}")
    print(f"Eval samples: {len(eval_samples)}")
    return all_samples, train_samples, eval_samples

def make_rejected_response(target_dict, rng, hard_negative=False):
    bad = deepcopy(target_dict)

    if hard_negative:
        # Small realistic mistakes
        bad["temp_max"] = round(float(bad["temp_max"]) + rng.uniform(-2.0, 2.0), 1)
        bad["temp_min"] = round(float(bad["temp_min"]) + rng.uniform(-2.0, 2.0), 1)
        bad["precipitation"] = round(max(0.0, float(bad["precipitation"]) + rng.uniform(-2.0, 2.0)), 1)
        bad["wind"] = round(max(0.0, float(bad["wind"]) + rng.uniform(-1.5, 1.5)), 1)

        if rng.random() < 0.35:
            other_labels = [x for x in WEATHER_LABELS if x != bad["weather"]]
            if other_labels:
                bad["weather"] = rng.choice(other_labels)
    else:
        # Bigger obvious mistakes
        corruption_types = rng.sample(
            ["temp_max", "temp_min", "precipitation", "wind", "weather", "swap_temps"],
            k=rng.randint(1, 3),
        )

        if "temp_max" in corruption_types:
            bad["temp_max"] = round(float(bad["temp_max"]) + rng.uniform(-8.0, 8.0), 1)

        if "temp_min" in corruption_types:
            bad["temp_min"] = round(float(bad["temp_min"]) + rng.uniform(-8.0, 8.0), 1)

        if "precipitation" in corruption_types:
            bad["precipitation"] = round(
                max(0.0, float(bad["precipitation"]) + rng.uniform(-10.0, 10.0)), 1
            )

        if "wind" in corruption_types:
            bad["wind"] = round(max(0.0, float(bad["wind"]) + rng.uniform(-5.0, 5.0)), 1)

        if "weather" in corruption_types:
            other_labels = [x for x in WEATHER_LABELS if x != bad["weather"]]
            if other_labels:
                bad["weather"] = rng.choice(other_labels)

        if "swap_temps" in corruption_types:
            bad["temp_max"], bad["temp_min"] = bad["temp_min"], bad["temp_max"]

    return json.dumps(bad, separators=(",", ":"))

def sft_to_preference_samples(sft_samples, seed=42, include_system=False):
    rng = random.Random(seed)
    preference_samples = []

    for sample in sft_samples:
        messages = sample.get("messages", [])

        system_msg = next((m["content"] for m in messages if m["role"] == "system"), "")
        user_msg = next((m["content"] for m in messages if m["role"] == "user"), None)
        assistant_msg = next((m["content"] for m in messages if m["role"] == "assistant"), None)

        if not user_msg or not assistant_msg:
            continue

        try:
            target_dict = json.loads(assistant_msg)
        except json.JSONDecodeError:
            continue

        prompt = user_msg if not include_system else f"System: {system_msg}\n\nUser: {user_msg}"

        # Easy negative
        rejected_easy = make_rejected_response(target_dict, rng, hard_negative=False)
        if rejected_easy != assistant_msg:
            preference_samples.append({
                "prompt": prompt,
                "chosen": assistant_msg,
                "rejected": rejected_easy,
            })

        # Hard negative
        rejected_hard = make_rejected_response(target_dict, rng, hard_negative=True)
        if rejected_hard != assistant_msg:
            preference_samples.append({
                "prompt": prompt,
                "chosen": assistant_msg,
                "rejected": rejected_hard,
            })

    return preference_samples

def write_preference_samples(train_sft_samples, eval_sft_samples):
    pref_train = sft_to_preference_samples(train_sft_samples, seed=42, include_system=False)
    pref_eval = sft_to_preference_samples(eval_sft_samples, seed=123, include_system=False)

    write_jsonl(PREF_TRAIN_OUT_PATH, pref_train)
    write_jsonl(PREF_EVAL_OUT_PATH, pref_eval)

    print(f"Train preference samples: {len(pref_train)}")
    print(f"Eval preference samples: {len(pref_eval)}")
    return pref_train, pref_eval

weather_data_path = pull_data("ananthr1/weather-prediction", DATA_DIR / "weather-prediction")
df = format_data(Path(weather_data_path) / EXTERNAL_DATASET)
all_samples, train_samples, eval_samples = write_samples(df)
pref_train_samples, pref_eval_samples = write_preference_samples(train_samples, eval_samples)

print(df.head())
print("\nSFT example:")
print(train_samples[0])

print("\nPreference example:")
print(pref_train_samples[0])

100%|██████████| 11.5k/11.5k [00:00<00:00, 16.9MB/s]

Extracting files...


Total samples: 5774
Train samples: 4618
Eval samples: 1156
Train preference samples: 9165
Eval preference samples: 2292
        date  precipitation  temp_max  temp_min  wind  weather
0 2012-01-01            0.0      12.8       5.0   4.7  drizzle
1 2012-01-02           10.9      10.6       2.8   4.5     rain
2 2012-01-03            0.8      11.7       7.2   2.3     rain
3 2012-01-04           20.3      12.2       5.6   4.7     rain
4 2012-01-05            1.3       8.9       2.8   6.1     rain

SFT example:
{'messages': [{'role': 'system', 'content': 'You are a weather forecasting assistant. You should only output in JSON format'}, {'role': 'user', 'content': 'Given the last 7 days of Seattle weather, predict tomorrow.\nDay -7 (2012-01-01): temp_max=12.8, temp_min=5.0, precipitation=0.0, wind=4.7, weather=drizzle\nDay -6 (2012-01-02): temp_max=10.6, temp_min=2.8, precipitation=10.9, wind=4.5, weather=rain\nDay -5 (2012-01-03): temp_max=11.7, temp_min=7.2, precipitation=0.8, wind=2.3, we

In [4]:
!pip -q install -U transformers datasets trl peft accelerate bitsandbytes unsloth

In [6]:
from datasets import load_dataset
from trl import RewardTrainer, RewardConfig
from peft import LoraConfig
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch

BASE_RM_MODEL = "Qwen/Qwen2-7B-Instruct"
PREF_TRAIN_OUT_PATH = "/content/weather_data/seattle_weather_pref_train.jsonl"
PREF_EVAL_OUT_PATH  = "/content/weather_data/seattle_weather_pref_eval.jsonl"
RM_OUTPUT_DIR = "/content/weather_reward_model"

dataset = load_dataset(
    "json",
    data_files={
        "train": PREF_TRAIN_OUT_PATH,
        "eval": PREF_EVAL_OUT_PATH,
    },
)

tokenizer = AutoTokenizer.from_pretrained(BASE_RM_MODEL, use_fast=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForSequenceClassification.from_pretrained(
    BASE_RM_MODEL,
    num_labels=1,
    dtype=(
        torch.bfloat16 if torch.cuda.is_available() and torch.cuda.is_bf16_supported()
        else torch.float16 if torch.cuda.is_available()
        else torch.float32
    ),
)
model.config.pad_token_id = tokenizer.pad_token_id

peft_config = LoraConfig(
    r=16,
    lora_alpha=16,
    lora_dropout=0.0,
    bias="none",
    task_type="SEQ_CLS",
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    modules_to_save=["score"],
)

training_args = RewardConfig(
    output_dir=RM_OUTPUT_DIR,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=4,
    num_train_epochs=2,
    learning_rate=1e-4,
    warmup_ratio=0.03,
    logging_steps=10,
    eval_strategy="epoch",
    save_strategy="epoch",
    save_total_limit=2,
    load_best_model_at_end=True,
    fp16=torch.cuda.is_available() and not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_available() and torch.cuda.is_bf16_supported(),
    report_to="none",
    max_length=2048,
    center_rewards_coefficient=1e-2,
)

trainer = RewardTrainer(
    model=model,
    processing_class=tokenizer,
    args=training_args,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    peft_config=peft_config,
)

trainer.train()

rm_adapter_dir = f"{RM_OUTPUT_DIR}-adapter"
trainer.model.save_pretrained(rm_adapter_dir)
tokenizer.save_pretrained(rm_adapter_dir)
print("Saved reward-model adapter to:", rm_adapter_dir)

Loading checkpoint shards:   0%|          | 0/4 [00:00<?, ?it/s]

Some weights of Qwen2ForSequenceClassification were not initialized from the model checkpoint at Qwen/Qwen2-7B-Instruct and are newly initialized: ['score.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Map:   0%|          | 0/9165 [00:00<?, ? examples/s]

Map:   0%|          | 0/9165 [00:00<?, ? examples/s]

Filter:   0%|          | 0/9165 [00:00<?, ? examples/s]

...

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                  ┃ rejected_text                                 ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ {"temp_max":13.9,"temp_min":6.1,"precipitat… │ {"temp_max":13.9,"temp_min":6.1,"precipitati… │ [0.7009, 0.2991] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.9,"temp_min":6.1,"precipitat… │ {"temp_max":13.5,"temp_min":4.5,"precipitati… │ [0.9952, 0.0048] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.3,"temp_min":4.4,"precipitat… │ {"temp_max":7.9,"temp_min":4.4,"precipitatio… │ [0.9944, 0.0056] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.3,"temp_min":4.4,"precipitat… │ {"temp_max":12.6,"temp_min":3.4,"precipitati… │ [0.9953, 0.0047] │
└──────────────────────────────────────────────┴───────────────────────────────────────────────┴──────────────────┘

/usr/local/lib/python3.12/dist-packages/torch/utils/checkpoint.py:232: UserWarning: None of the inputs have requires_grad=True. Gradients will be None
  check_backward_validity(args)


┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━┓
┃ chosen_text                                  ┃ rejected_text                                 ┃ logits           ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━┩
│ {"temp_max":13.9,"temp_min":6.1,"precipitat… │ {"temp_max":13.9,"temp_min":6.1,"precipitati… │ [0.7677, 0.2323] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.9,"temp_min":6.1,"precipitat… │ {"temp_max":13.5,"temp_min":4.5,"precipitati… │ [0.9961, 0.0039] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.3,"temp_min":4.4,"precipitat… │ {"temp_max":7.9,"temp_min":4.4,"precipitatio… │ [0.9958, 0.0042] │
├──────────────────────────────────────────────┼───────────────────────────────────────────────┼──────────────────┤
│ {"temp_max":13.3,"temp_min":4.4,"precipitat… │ {"temp_max":12.6,"temp_min":3.4,"precipitati… │ [0.9952, 0.0048] │
└──────────────────────────────────────────────┴───────────────────────────────────────────────┴──────────────────┘

Saved reward-model adapter to: /content/weather_reward_model-adapter


In [7]:
import shutil
shutil.make_archive(rm_adapter_dir, "zip", rm_adapter_dir)

'/content/weather_reward_model-adapter.zip'

In [4]:
from datasets import load_dataset
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=MODEL_NAME,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
)

raw_datasets = load_dataset(
    "json",
    data_files={
        "train": str(TRAIN_OUT_PATH),
        "eval": str(EVAL_OUT_PATH),
    },
)

def apply_template(batch):
    texts = []
    for messages in batch["messages"]:
        text = tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False,
        )
        texts.append(text)
    return {"text": texts}

dataset = raw_datasets.map(apply_template, batched=True)
dataset

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2026.4.2: Fast Qwen2 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-80GB. Num GPUs = 1. Max memory: 79.251 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/266 [00:00<?, ?B/s]

...

Generating train split: 0 examples [00:00, ? examples/s]

Generating eval split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/4618 [00:00<?, ? examples/s]

Map:   0%|          | 0/1156 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['messages', 'text'],
        num_rows: 4618
    })
    eval: Dataset({
        features: ['messages', 'text'],
        num_rows: 1156
    })
})

In [5]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=67,
    use_rslora=False,
    loftq_config=None,
)

Unsloth 2026.4.2 patched 28 layers with 28 QKV layers, 28 O layers and 28 MLP layers.


In [6]:
from trl import SFTTrainer
from transformers import TrainingArguments, EarlyStoppingCallback

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset["train"],
    eval_dataset=dataset["eval"],
    dataset_text_field="text",
    max_seq_length=MAX_SEQ_LENGTH,
    packing=False,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    args=TrainingArguments(
        output_dir=OUTPUT_DIR,
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        num_train_epochs=NUM_TRAIN_EPOCHS,
        learning_rate=2e-4,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.01,
        lr_scheduler_type="linear",
        seed=67,
        fp16=False,
        bf16=True,
        report_to="none",
        eval_strategy="epoch",
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
    ),
)

trainer_stats = trainer.train()
trainer_stats

adapter_dir = f"{OUTPUT_DIR}-adapter"
model.save_pretrained(adapter_dir)
tokenizer.save_pretrained(adapter_dir)
print(f"Saved adapter to: {adapter_dir}")

import shutil
shutil.make_archive(adapter_dir, "zip", adapter_dir)

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/4618 [00:00<?, ? examples/s]

Unsloth: Tokenizing ["text"] (num_proc=16):   0%|          | 0/1156 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': None}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 4,618 | Num Epochs = 5 | Total steps = 2,890
O^O/ \_/ \    Batch size per device = 2 | Gradient accumulation steps = 4
\        /    Data Parallel GPUs = 1 | Total batch size (2 x 4 x 1) = 8
 "-____-"     Trainable parameters = 40,370,176 of 7,655,986,688 (0.53% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


Epoch,Training Loss,Validation Loss
1,0.027662,0.621364
2,0.017811,0.726351
3,0.012239,0.795396


/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:172: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API 

Saved adapter to: sft-lora-adapter


'/content/sft-lora-adapter.zip'

In [7]:
FastLanguageModel.for_inference(model)

sample_prompt = {
    "messages": [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": """Given the last 7 days of Seattle weather, predict tomorrow.
Day -7 (2012-01-01): temp_max=12.8, temp_min=5.0, precipitation=0.0, wind=4.7, weather=drizzle
Day -6 (2012-01-02): temp_max=10.6, temp_min=2.8, precipitation=10.9, wind=4.5, weather=rain
Day -5 (2012-01-03): temp_max=11.7, temp_min=7.2, precipitation=0.8, wind=2.3, weather=rain
Day -4 (2012-01-04): temp_max=12.2, temp_min=5.6, precipitation=20.3, wind=4.7, weather=rain
Day -3 (2012-01-05): temp_max=8.9, temp_min=2.8, precipitation=1.3, wind=6.1, weather=rain
Day -2 (2012-01-06): temp_max=4.4, temp_min=2.2, precipitation=2.5, wind=2.2, weather=rain
Day -1 (2012-01-07): temp_max=7.2, temp_min=2.8, precipitation=0.0, wind=2.3, weather=rain
Return JSON with keys: temp_max, temp_min, precipitation, wind, weather."""},
    ]
}

prompt_text = tokenizer.apply_chat_template(
    sample_prompt["messages"],
    tokenize=False,
    add_generation_prompt=True,
)

inputs = tokenizer(prompt_text, return_tensors="pt").to(model.device)
outputs = model.generate(
    **inputs,
    max_new_tokens=128,
    temperature=0.7,
    do_sample=False,
)

print(tokenizer.decode(outputs[0], skip_special_tokens=True))

Both `max_new_tokens` (=128) and `max_length`(=32768) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:71: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)
/usr/local/lib/python3.12/dist-packages/transformers/modeling_attn_mask_utils.py:281: FutureWarning: The attention mask API under `transformers.modeling_attn_mask_utils` (`AttentionMaskConverter`) is deprecated and will be removed in Transformers v5.10. Please use the new API in `transformers.masking_utils`.
  warnings.warn(DEPRECATION_MESSAGE, FutureWarning)


system
You are a weather forecasting assistant. You should only output in JSON format
user
Given the last 7 days of Seattle weather, predict tomorrow.
Day -7 (2012-01-01): temp_max=12.8, temp_min=5.0, precipitation=0.0, wind=4.7, weather=drizzle
Day -6 (2012-01-02): temp_max=10.6, temp_min=2.8, precipitation=10.9, wind=4.5, weather=rain
Day -5 (2012-01-03): temp_max=11.7, temp_min=7.2, precipitation=0.8, wind=2.3, weather=rain
Day -4 (2012-01-04): temp_max=12.2, temp_min=5.6, precipitation=20.3, wind=4.7, weather=rain
Day -3 (2012-01-05): temp_max=8.9, temp_min=2.8, precipitation=1.3, wind=6.1, weather=rain
Day -2 (2012-01-06): temp_max=4.4, temp_min=2.2, precipitation=2.5, wind=2.2, weather=rain
Day -1 (2012-01-07): temp_max=7.2, temp_min=2.8, precipitation=0.0, wind=2.3, weather=rain
Return JSON with keys: temp_max, temp_min, precipitation, wind, weather.
assistant
{"temp_max":10.0,"temp_min":2.8,"precipitation":0.0,"wind":2.0,"weather":"sun"}
